# 7. The Simulated OIDC Dance: Putting It All Together

Every previous notebook built one ingredient. This notebook combines them into the
actual **choreography** of a modern SSO login, the "OIDC dance", using plain
Python objects standing in for the real actors, so you can see the full sequence
without HTTP, browsers, or network requests getting in the way. Part 2 of this
repository rebuilds this exact flow as real, running web servers.

By the end of this notebook you should be able to answer:

- What are the roles of the **Identity Provider**, the **Relying Party**, and the
  **Browser/User**, and how does this map onto "I log into my app using Okta"?
- What gets exchanged at each step, and which key signs or verifies it?
- Where would this break if the Relying Party skipped a verification step?

## 7.1 Mapping the cast of characters

If you use Okta at work, or explored Auth0 for your Raspberry Pi projects, this
maps directly onto what you already do:

| Role in this notebook | What it represents | Real-world example |
|---|---|---|
| **Identity Provider (IdP)** | The service that knows who you are and can prove it | Okta, Auth0, Google, your company's SSO |
| **Relying Party (RP)** | The app you're trying to log into | Your app on the Raspberry Pi, or any "Sign in with..." button |
| **Browser / User** | You, sitting between the two, redirected back and forth | You, clicking "Log in" |

The entire point of this design is that the **Relying Party never sees your
password**. Only the Identity Provider does. The RP only ever receives a signed
token telling it who you are. This is exactly why "Sign in with Google/Okta/etc."
buttons don't ask you to type your Google/Okta password into the app you're
logging into.

## 7.2 Setting up the Identity Provider

The IdP holds a private signing key, a small "user directory" (hardcoded here for
simplicity; Notebook 3's salted-hash technique is what a real one would use
internally), and issues authorization codes and signed tokens.

In [1]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
import json, base64, time, secrets, hashlib, os

def b64url_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")

def b64url_decode(s: str) -> bytes:
    padding_needed = 4 - (len(s) % 4)
    if padding_needed != 4:
        s += "=" * padding_needed
    return base64.urlsafe_b64decode(s)


class IdentityProvider:
    """Stands in for Okta / Auth0 / any OIDC-compliant Identity Provider."""

    def __init__(self, issuer_url: str):
        self.issuer_url = issuer_url
        self._private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
        self.public_key = self._private_key.public_key()  # this is what the IdP "publishes"

        # A tiny hardcoded user directory. Passwords are salted+hashed (Notebook 3's technique).
        self._users = {}
        self._register_user("alice", "correct-horse-battery-staple", "Alice Example", "alice@example.com")

        self._pending_codes = {}  # authorization_code -> username (single-use, short-lived)

    def _register_user(self, username, password, name, email):
        salt = os.urandom(16)
        digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 200_000)
        self._users[username] = {"salt": salt, "hash": digest, "name": name, "email": email}

    def check_credentials(self, username: str, password: str) -> bool:
        user = self._users.get(username)
        if not user:
            return False
        attempt = hashlib.pbkdf2_hmac("sha256", password.encode(), user["salt"], 200_000)
        return attempt == user["hash"]

    def issue_authorization_code(self, username: str) -> str:
        """Called after a successful login. Returns a short-lived, single-use code."""
        code = secrets.token_urlsafe(24)
        self._pending_codes[code] = {"username": username, "expires": time.time() + 60}
        return code

    def exchange_code_for_token(self, code: str) -> str:
        """The 'token endpoint' -- called server-to-server by the Relying Party, not the browser."""
        entry = self._pending_codes.pop(code, None)
        if entry is None:
            raise ValueError("Invalid or already-used authorization code")
        if entry["expires"] < time.time():
            raise ValueError("Authorization code expired")

        user = self._users[entry["username"]]
        now = int(time.time())
        payload = {
            "iss": self.issuer_url,
            "sub": entry["username"],
            "name": user["name"],
            "email": user["email"],
            "iat": now,
            "exp": now + 3600,
        }
        header = {"alg": "RS256", "typ": "JWT"}
        header_b64 = b64url_encode(json.dumps(header, separators=(",", ":")).encode())
        payload_b64 = b64url_encode(json.dumps(payload, separators=(",", ":")).encode())
        signing_input = f"{header_b64}.{payload_b64}".encode()
        signature = self._private_key.sign(signing_input, padding.PKCS1v15(), hashes.SHA256())
        return f"{header_b64}.{payload_b64}.{b64url_encode(signature)}"


idp = IdentityProvider(issuer_url="https://idp.example.com")
print("Identity Provider ready. Public key can be freely distributed:")
print(idp.public_key)


Identity Provider ready. Public key can be freely distributed:


## 7.3 Setting up the Relying Party

The Relying Party only needs the IdP's **public key** (never the private key) to
independently verify tokens it receives. It never has to "call back" the IdP to
ask "is this real?" once it has that public key.

In [ ]:
## 7.3 Setting up the Relying Party

The Relying Party only needs the IdP's **public key** (never the private key) to
independently verify tokens it receives. It never has to "call back" the IdP to
ask "is this real?" once it has that public key.

Relying Party ready -- holds only the IdP's PUBLIC key.


## 7.4 Running the full dance

Here's the sequence a real "Sign in with Okta" button triggers, narrated step by
step. In Part 2 of this repository, every one of these steps becomes a real HTTP
request between two actual servers, but the logic is identical to what's below.

In [ ]:
print("STEP 1 -- Browser/User clicks 'Log in' on the Relying Party's app.")
print("STEP 2 -- Relying Party redirects the browser to the Identity Provider.")
print("          (In real OIDC, this is a redirect to the IdP's /authorize endpoint,")
print("           carrying a client_id, redirect_uri, and a random 'state' value.)")
print()

print("STEP 3 -- Identity Provider prompts for credentials. User enters them.")
username, password = "alice", "correct-horse-battery-staple"
if idp.check_credentials(username, password):
    print("          Credentials verified against the salted, hashed user directory.")
else:
    raise SystemExit("Login failed")
print()

print("STEP 4 -- Identity Provider issues a short-lived, single-use authorization code")
print("          and redirects the browser back to the Relying Party with it.")
auth_code = idp.issue_authorization_code(username)
print("          Authorization code:", auth_code)
print()

print("STEP 5 -- Relying Party's SERVER (not the browser) exchanges the code for a token.")
print("          It calls the Identity Provider directly over a back-channel connection.")
id_token = idp.exchange_code_for_token(auth_code)
print("          Received signed token:")
print("         ", id_token[:60], "...")
print()

print("STEP 6 -- Relying Party verifies the token's signature using the IdP's PUBLIC key,")
print("          checks expiry and issuer, and only THEN considers the user logged in.")
claims = relying_party.verify_token(id_token)
print("          ✅ Verified identity:", claims)

STEP 1 -- Browser/User clicks 'Log in' on the Relying Party's app.
STEP 2 -- Relying Party redirects the browser to the Identity Provider.
          (In real OIDC, this is a redirect to the IdP's /authorize endpoint,
           carrying a client_id, redirect_uri, and a random 'state' value.)

STEP 3 -- Identity Provider prompts for credentials. User enters them.
          Credentials verified against the salted, hashed user directory.

STEP 4 -- Identity Provider issues a short-lived, single-use authorization code
          and redirects the browser back to the Relying Party with it.
          Authorization code: Z1R3YXlEnzRHulQstmlnLJA-oejB5WrN

STEP 5 -- Relying Party's SERVER (not the browser) exchanges the code for a token,
          calling the Identity Provider directly over a back-channel connection.
          Received signed token:
          eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL2l ...

STEP 6 -- Relying Party verifies the token's signature using the IdP's P

## 7.5 Where this would break, if a step were skipped

- **If Step 6 skipped signature verification**: the Relying Party would trust
  *any* JSON payload handed to it, including one an attacker fabricated. This is
  the single most important check in the whole flow.
- **If the authorization code in Step 4 weren't single-use**: an attacker who
  intercepted it (e.g., via browser history, a referrer header, or a malicious
  redirect) could reuse it to obtain their own valid token for someone else's
  identity.
- **If the Relying Party never checked `exp`**: a token stolen long after issuance
  would remain valid forever.
- **If the Relying Party accepted tokens from any `iss`, not just its trusted
  one**: an attacker who ran their *own* Identity Provider, with its own private
  key, could sign tokens claiming to be anyone. The Relying Party would have
  no way to tell it wasn't the real IdP.

Every one of these is a real, named category of vulnerability in production OIDC
implementations. This is exactly why using an established library (or an
established provider like Okta or Auth0) rather than hand-rolling this in
production is almost always the right call. This repository builds it by hand
purely so you can see what that library is doing for you.

## Summary: the whole repository in one paragraph

An Identity Provider holds a private key it never shares, and uses it to
**sign** tokens after verifying a user's password (itself stored as a salted
hash, never in plaintext). A Relying Party holds only the IdP's **public** key,
and uses it purely to **verify** (never to sign anything itself). The token
passed between them is a JWT: readable by anyone, but only forgeable by whoever
holds the private key. The entire security of the system rests on that one private
key staying private, and on the Relying Party performing every verification step
faithfully. That's it - that's the whole trick.

**Next:** `part2-web-app/` - the same flow, rebuilt as two real Flask servers
communicating over actual HTTP, with a live process trace so you can watch every
step happen in your browser.